# 📝 ROTBOTS — TikTok-Style Subtitles

---

Word-by-word animated subtitles from narration audio using Whisper.

5 styles — choose one or mix randomly.

> **🤔** How do subtitles change how we consume video?

---

In [ ]:
# SETUP
!pip install -q faster-whisper
import json, random, subprocess
from pathlib import Path
from IPython.display import display, Markdown
from google.colab import drive
drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
print('✅ Setup complete')

In [ ]:
# SELECT SESSION
sessions = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir() and (d/'session_info.json').exists()])
if not sessions: print('❌ No sessions!')
else:
    print('📂 Sessions:')
    for i,s in enumerate(sessions): print(f'   {i}: {s}')
SESSION_NAME = sessions[-1] if sessions else ''
SESSION_DIR = BASE_DIR / SESSION_NAME
AUDIO_DIR = SESSION_DIR / 'audio'
print(f'✅ Session: {SESSION_NAME}')

In [ ]:
# ============================================================
# SUBTITLE SETTINGS
# ============================================================
# Styles: '01'=Impact white, '04'=Purple outline, '05'=Dark box,
#         '06'=Orange bouncy, '08'=Neon yellow-pink
# Or 'random' to mix all styles

SUBTITLE_MODE = 'random'  # Options: 'random', '01', '04', '05', '06', '08'
MIXUP_MIN_SECS = 10
MIXUP_MAX_SECS = 25

print(f'📝 Mode: {SUBTITLE_MODE}')

In [ ]:
# STYLE DEFINITIONS (from original ROTBOTS)
STYLES = {
    '01': {'font':'Impact','size':80,'bold':-1,'primary':'&H00FFFFFF','outline_c':'&H00000000','back':'&H00000000','border_style':1,'outline':7,'shadow':0,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,40,\fscx130\fscy130)\t(40,80,\fscx100\fscy100)}','desc':'Classic Impact (white, bounce)'},
    '04': {'font':'Arial Black','size':76,'bold':-1,'primary':'&H00FFFFFF','outline_c':'&H008030FF','back':'&H00000000','border_style':1,'outline':8,'shadow':3,'alignment':5,'margin_v':30,'upper':True,'anim':r'{\fscx0\fscy0\t(0,30,\fscx140\fscy140)\t(30,70,\fscx100\fscy100)}','desc':'Purple outline (bounce)'},
    '05': {'font':'Arial Black','size':70,'bold':0,'primary':'&H00FFFFFF','outline_c':'&H00000000','back':'&HCC1A1A1A','border_style':3,'outline':16,'shadow':0,'alignment':2,'margin_v':80,'upper':True,'anim':r'{\fad(80,0)\fscx95\fscy95\t(0,60,\fscx100\fscy100)}','desc':'Dark box (fade-in)'},
    '06': {'font':'Impact','size':85,'bold':0,'primary':'&H0000DDFF','outline_c':'&H000000CC','back':'&H00000000','border_style':1,'outline':7,'shadow':4,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,25,\fscx150\fscy150)\t(25,55,\fscx90\fscy90)\t(55,80,\fscx105\fscy105)\t(80,100,\fscx100\fscy100)}','desc':'Orange spring (bouncy)'},
    '08': {'font':'Arial Black','size':72,'bold':0,'primary':'&H00FFFF00','outline_c':'&H00FF0066','back':'&H00000000','border_style':1,'outline':5,'shadow':5,'alignment':5,'margin_v':20,'upper':True,'anim':r'{\fscx0\fscy0\t(0,40,\fscx115\fscy115)\t(40,80,\fscx100\fscy100)}','desc':'Neon yellow-pink (pop-in)'},
}
print('🎨 Styles:')
for k,v in STYLES.items(): print(f'   {k}: {v["desc"]}')

In [ ]:
# WHISPER: Extract word timestamps
from faster_whisper import WhisperModel
print('🧠 Loading Whisper...')
whisper_model = WhisperModel('base', device='cpu', compute_type='int8')
print('✅ Loaded')

audio_files = sorted(AUDIO_DIR.glob('narration_*.mp3')) if AUDIO_DIR.exists() else []
print(f'🎙️ {len(audio_files)} narration files')

all_words = []
time_offset = 0.0
for af in audio_files:
    print(f'   {af.name}...', end=' ', flush=True)
    segs, info = whisper_model.transcribe(str(af), word_timestamps=True, language='en')
    fwords = []
    for seg in segs:
        if seg.words:
            for w in seg.words:
                fwords.append({'word':w.word.strip(),'start':w.start+time_offset,'end':w.end+time_offset})
    try:
        r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(af)],capture_output=True,text=True,timeout=10)
        fdur = float(r.stdout.strip())
    except: fdur = 5.0
    time_offset += fdur
    all_words.extend(fwords)
    print(f'{len(fwords)} words')
print(f'\n✅ {len(all_words)} words, {time_offset:.1f}s')

In [ ]:
# GENERATE ASS FILE
W=1280; H=720; scale=H/512

def style_line(name, s):
    sz=int(s['size']*scale); ol=int(s['outline']*scale); mv=int(s['margin_v']*scale)
    return f'Style: {name},{s["font"]},{sz},{s["primary"]},&H0000FFFF,{s["outline_c"]},{s["back"]},{s["bold"]},0,0,0,100,100,2,0,{s["border_style"]},{ol},{s["shadow"]},{s["alignment"]},10,10,{mv},1'

def fmt(seconds):
    h=int(seconds//3600);m=int((seconds%3600)//60);s=int(seconds%60);cs=int((seconds%1)*100)
    return f'{h}:{m:02d}:{s:02d}.{cs:02d}'

mixup = SUBTITLE_MODE == 'random'
if mixup:
    sl = [style_line(f'TT{k}',v) for k,v in STYLES.items()]
else:
    sk = SUBTITLE_MODE if SUBTITLE_MODE in STYLES else '01'
    sl = [style_line('TT', STYLES[sk])]

header = f'[Script Info]\nTitle: ROTBOTS Subtitles\nScriptType: v4.00+\nWrapStyle: 0\nScaledBorderAndShadow: yes\nPlayResX: {W}\nPlayResY: {H}\n\n[V4+ Styles]\nFormat: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n' + '\n'.join(sl) + '\n\n[Events]\nFormat: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n'

keys = list(STYLES.keys())
ck = random.choice(keys) if mixup else (SUBTITLE_MODE if SUBTITLE_MODE in STYLES else '01')
ns = random.uniform(MIXUP_MIN_SECS, MIXUP_MAX_SECS) if mixup else 99999

dlg = []
for w in all_words:
    if mixup and w['start'] >= ns:
        ck = random.choice(keys)
        ns = w['start'] + random.uniform(MIXUP_MIN_SECS, MIXUP_MAX_SECS)
    sty = STYLES[ck]
    sn = f'TT{ck}' if mixup else 'TT'
    wd = w['word'].replace('\\','\\\\').replace('{','\\{').replace('}','\\}')
    text = sty['anim'] + (wd.upper() if sty['upper'] else wd)
    dlg.append(f'Dialogue: 0,{fmt(w["start"])},{fmt(w["end"])},{sn},,0,0,0,,{text}')

ass_path = SESSION_DIR / 'subtitles.ass'
with open(ass_path, 'w', encoding='utf-8') as f:
    f.write(header + '\n'.join(dlg))

print(f'✅ {ass_path.name}: {len(dlg)} words')
print(f'   Mode: {"random mix" if mixup else STYLES.get(ck,{}).get("desc",ck)}')
print(f'\n→ Run 06_Assemble next — it will detect subtitles.ass automatically!')

---
*ROTBOTS Workshop — LI-MA 2026*